In [ ]:
# ============================================================
# COLAB TRANSFER - 03: PREVIEW E PUBLICAÇÃO DO ESTADO COMPLETO
# ============================================================
# O Dataset remoto só muda após preview e confirmação explícita.

import json
import os
import subprocess
import sys
from pathlib import Path

WORKDIR = Path('/content/colab-pipeline')
SCRIPTS_DIR = WORKDIR / 'scripts'
REPO_URL = 'https://github.com/automadevs/colab-pipeline.git'
if WORKDIR.exists():
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)], check=True)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

MANAGER = SCRIPTS_DIR / 'kaggle_dataset_manager.py'
if not MANAGER.exists():
    raise FileNotFoundError(f'Módulo não encontrado: {MANAGER}')

# Autenticação Kaggle: Secret/env, sem imprimir credenciais.
def ensure_kaggle_auth():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        return
    username = os.environ.get('KAGGLE_USERNAME')
    key = os.environ.get('KAGGLE_KEY')
    try:
        from google.colab import userdata
        username = username or userdata.get('KAGGLE_USERNAME')
        key = key or userdata.get('KAGGLE_KEY')
    except Exception:
        pass
    if not username or not key:
        raise RuntimeError('KAGGLE_USERNAME/KAGGLE_KEY não configurados nos Secrets/env')
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_text(json.dumps({'username': username, 'key': key}))
    os.chmod(kaggle_json, 0o600)

ensure_kaggle_auth()
from kaggle_dataset_manager import publish_staged_state

DATASET = 'automamermaid/comfydocs'
STAGING_DIR = Path('/content/kaggle_staging')
if not STAGING_DIR.exists():
    raise FileNotFoundError(f'Staging não encontrado: {STAGING_DIR}. Execute 02_download primeiro.')

version_output = publish_staged_state(DATASET, STAGING_DIR)
if version_output is None:
    print('Publicação cancelada; Dataset remoto não foi alterado.')
else:
    print('\nDATASET UPDATE COMPLETE')
    print(f'Dataset: {DATASET}')
    print(version_output)